# Lawrence, MA — Demographics Walkthrough
**Census ACS 5-Year 2022 · Essex County, MA**

This notebook demonstrates how to load, decode, and visualise ACS demographic data
produced by the `census_downloader` pipeline. It covers:

- Loading CSVs and codebooks
- City-level key metrics
- Age pyramid
- Race / ethnicity distribution
- Hispanic / Latino origin breakdown
- Language spoken at home
- Census-tract level comparisons

**Prerequisites:** generate the data files first (takes ~30 s):
```bash
python main.py --city lawrence --topic demographics --year 2022 --geo all
python main.py --topic demographics --codebook
```

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:,.1f}".format)

## 1. Load Data

In [2]:
ROOT     = Path(".")
DATA_DIR = ROOT / "outputs" / "lawrence" / "demographics" / "2022"
CB_PATH  = ROOT / "outputs" / "codebooks" / "demographics_codebook.csv"

place    = pd.read_csv(DATA_DIR / "place.csv")
tract    = pd.read_csv(DATA_DIR / "tract.csv")
bg       = pd.read_csv(DATA_DIR / "block_group.csv")
codebook = pd.read_csv(CB_PATH)

# Build code → label mapping for renaming columns
code_to_label = dict(zip(codebook["variable_code"], codebook["label"]))

print(f"Place rows:      {len(place)}")
print(f"Tract rows:      {len(tract)}")
print(f"Block-group rows:{len(bg)}")
print(f"Codebook entries:{len(codebook)}\n")
display(codebook[["variable_code", "sub_theme", "label"]].head(8))

Place rows:      1
Tract rows:      18
Block-group rows:56
Codebook entries:165



,variable_code,sub_theme,label
0,B01001_001E,Population & Age,Total population
1,B01002_001E,Population & Age,Median age (both sexes)
2,B01002_002E,Population & Age,Median age (male)
3,B01002_003E,Population & Age,Median age (female)
4,B01001_002E,Population & Age,Male: Total
5,B01001_003E,Population & Age,Male: Under 5 years
6,B01001_004E,Population & Age,Male: 5 to 9 years
7,B01001_005E,Population & Age,Male: 10 to 14 years


## 2. City-Level Key Metrics

In [3]:
city = place.iloc[0]

total_pop    = int(city["B01001_001E"])
median_age   = city["B01002_001E"]
hisp_n       = city["B03001_003E"]
hisp_t       = city["B03001_001E"]
no_eng       = (city["B06007_003E"] + city["B06007_005E"]
                + city["B06007_007E"] + city["B06007_009E"])
lang_total   = city["B06007_001E"]
not_citizen  = city["B05001_006E"]
citizen_total= city["B05001_001E"]

print("=" * 50)
print("  Lawrence, MA  ·  ACS 5-Year 2022")
print("=" * 50)
print(f"  Total population:         {total_pop:>10,}")
print(f"  Median age:               {median_age:>10.1f}")
print(f"  Hispanic/Latino:          {hisp_n:>10,.0f}  ({hisp_n/hisp_t*100:.1f}%)")
print(f"  Non-English at home:      {no_eng:>10,.0f}  ({no_eng/lang_total*100:.1f}%)")
print(f"  Non-citizen (est.):       {not_citizen:>10,.0f}  ({not_citizen/citizen_total*100:.1f}%)")
print("=" * 50)

  Lawrence, MA  ·  ACS 5-Year 2022
  Total population:             88,067
  Median age:                     31.5
  Hispanic/Latino:              72,179  (82.0%)
  Non-English at home:         121,638  (149.8%)
  Non-citizen (est.):           20,667  (23.5%)


## 3. Age Pyramid

In [4]:
# Adjacent Census age bands merged into standard 5-year groups
AGE_GROUPS = [
    ("Under 5", ["B01001_003E"],                                       ["B01001_027E"]),
    ("5–9",     ["B01001_004E"],                                       ["B01001_028E"]),
    ("10–14",   ["B01001_005E"],                                       ["B01001_029E"]),
    ("15–19",   ["B01001_006E","B01001_007E"],                         ["B01001_030E","B01001_031E"]),
    ("20–24",   ["B01001_008E","B01001_009E","B01001_010E"],           ["B01001_032E","B01001_033E","B01001_034E"]),
    ("25–29",   ["B01001_011E"],                                       ["B01001_035E"]),
    ("30–34",   ["B01001_012E"],                                       ["B01001_036E"]),
    ("35–39",   ["B01001_013E"],                                       ["B01001_037E"]),
    ("40–44",   ["B01001_014E"],                                       ["B01001_038E"]),
    ("45–49",   ["B01001_015E"],                                       ["B01001_039E"]),
    ("50–54",   ["B01001_016E"],                                       ["B01001_040E"]),
    ("55–59",   ["B01001_017E"],                                       ["B01001_041E"]),
    ("60–64",   ["B01001_018E","B01001_019E"],                         ["B01001_042E","B01001_043E"]),
    ("65–69",   ["B01001_020E","B01001_021E"],                         ["B01001_044E","B01001_045E"]),
    ("70–74",   ["B01001_022E"],                                       ["B01001_046E"]),
    ("75–79",   ["B01001_023E"],                                       ["B01001_047E"]),
    ("80–84",   ["B01001_024E"],                                       ["B01001_048E"]),
    ("85+",     ["B01001_025E"],                                       ["B01001_049E"]),
]

labels, males, females = [], [], []
for lbl, m_codes, f_codes in AGE_GROUPS:
    labels.append(lbl)
    males.append(sum(city.get(c, 0) for c in m_codes))
    females.append(sum(city.get(c, 0) for c in f_codes))

max_v = max(max(males), max(females))
step  = next(s for s in [500, 1000, 2000, 2500, 5000] if max_v / s <= 6)
ticks = list(range(-round(max_v * 1.2 // step) * step,
                    round(max_v * 1.2 // step) * step + step, step))

fig = go.Figure()
fig.add_trace(go.Bar(
    name="Male",   y=labels, x=[-v for v in males],  orientation="h",
    marker_color="#1B3A6B", marker_line_width=0,
    customdata=males,
    hovertemplate="<b>%{y}</b><br>Male: %{customdata:,}<extra></extra>",
))
fig.add_trace(go.Bar(
    name="Female", y=labels, x=females, orientation="h",
    marker_color="#BE2D2C", marker_line_width=0,
    hovertemplate="<b>%{y}</b><br>Female: %{x:,}<extra></extra>",
))
fig.update_layout(
    title="Age Pyramid · Lawrence, MA · ACS 5-Year 2022",
    barmode="overlay", bargap=0.1,
    xaxis=dict(tickvals=ticks, ticktext=[f"{abs(v):,}" for v in ticks],
               title="Population", zeroline=True, zerolinewidth=2,
               zerolinecolor="#999", gridcolor="#efefef"),
    yaxis=dict(title="Age Group"),
    legend=dict(orientation="h", y=1.05, x=0.5, xanchor="center"),
    plot_bgcolor="white", paper_bgcolor="white",
    height=580, width=750,
    margin=dict(l=80, r=40, t=80, b=60),
)
fig.show()

## 4. Race Distribution

In [5]:
RACE_VARS = {
    "B02001_002E": "White alone",
    "B02001_003E": "Black or African American",
    "B02001_004E": "Amer. Indian & Alaska Native",
    "B02001_005E": "Asian",
    "B02001_006E": "Native Hawaiian & Pacific Isl.",
    "B02001_007E": "Some other race",
    "B02001_008E": "Two or more races",
}
RACE_COLORS = ["#1B3A6B","#BE2D2C","#D4A724","#2A9D8F","#6B4C8C","#CA7000","#52796F"]

total_race = city["B02001_001E"]
data_r = sorted(
    [(lbl, city[code]) for code, lbl in RACE_VARS.items()],
    key=lambda x: x[1], reverse=True,
)
labels_r, vals_r = zip(*data_r)
pcts_r = [v / total_race * 100 for v in vals_r]

fig = go.Figure(go.Bar(
    y=list(labels_r), x=list(vals_r), orientation="h",
    marker_color=RACE_COLORS[:len(labels_r)], marker_line_width=0,
    text=[f"{p:.1f}%" for p in pcts_r], textposition="outside",
    hovertemplate="<b>%{y}</b><br>%{x:,}  (%{text})<extra></extra>",
))
fig.update_layout(
    title="Race Distribution · Lawrence, MA · ACS 2022",
    xaxis=dict(title="Population", gridcolor="#efefef"),
    yaxis=dict(autorange="reversed"),
    plot_bgcolor="white", paper_bgcolor="white",
    height=380, width=740,
    margin=dict(l=225, r=100, t=70, b=50),
)
fig.show()

display(
    pd.DataFrame({
        "Group":      labels_r,
        "Population": [f"{int(v):,}" for v in vals_r],
        "Share":      [f"{p:.1f}%" for p in pcts_r],
    }).reset_index(drop=True)
)

,Group,Population,Share
0,Some other race,"45,192",51.3%
1,White alone,"22,956",26.1%
2,Two or more races,"13,277",15.1%
3,Black or African American,"4,535",5.1%
4,Asian,"1,852",2.1%
5,Amer. Indian & Alaska Native,255,0.3%
6,Native Hawaiian & Pacific Isl.,0,0.0%


## 5. Hispanic / Latino Origin

Lawrence is one of the most Hispanic cities in Massachusetts (~82 % of residents).
This chart breaks that population down by specific origin.

In [6]:
HISP_VARS = {
    "B03001_005E": "Puerto Rican",
    "B03001_007E": "Dominican",
    "B03001_008E": "Central American",
    "B03001_004E": "Mexican",
    "B03001_016E": "South American",
    "B03001_006E": "Cuban",
    "B03001_027E": "Other Hispanic/Latino",
    "B03001_002E": "Not Hispanic/Latino",
}
HISP_COLORS = ["#2A9D8F","#BE2D2C","#D4A724","#CA7000","#6B4C8C","#52796F","#4A5568","#1B3A6B"]

total_h = city["B03001_001E"]
data_h = sorted(
    [(lbl, city[code]) for code, lbl in HISP_VARS.items()],
    key=lambda x: x[1], reverse=True,
)
labels_h, vals_h = zip(*data_h)
pcts_h = [v / total_h * 100 for v in vals_h]

fig = go.Figure(go.Bar(
    y=list(labels_h), x=list(vals_h), orientation="h",
    marker_color=HISP_COLORS[:len(labels_h)], marker_line_width=0,
    text=[f"{p:.1f}%" for p in pcts_h], textposition="outside",
    hovertemplate="<b>%{y}</b><br>%{x:,}  (%{text})<extra></extra>",
))
fig.update_layout(
    title="Hispanic/Latino Ethnicity · Lawrence, MA · ACS 2022",
    xaxis=dict(title="Population", gridcolor="#efefef"),
    yaxis=dict(autorange="reversed"),
    plot_bgcolor="white", paper_bgcolor="white",
    height=390, width=740,
    margin=dict(l=200, r=110, t=70, b=50),
)
fig.show()

print(f"Total Hispanic/Latino: {city['B03001_003E']:,.0f}"
      f"  ({city['B03001_003E']/total_h*100:.1f}% of population)")

Total Hispanic/Latino: 72,179  (82.0% of population)


## 6. Language Spoken at Home

In [7]:
LANG_VARS = [
    ("B06007_002E", "English only"),
    ("B06007_003E", "Spanish"),
    ("B06007_005E", "Other Indo-European"),
    ("B06007_007E", "Asian & Pacific Island"),
    ("B06007_009E", "Other languages"),
]
LANG_COLORS = ["#1B3A6B","#BE2D2C","#2A9D8F","#D4A724","#6B4C8C"]

total_lang = city["B06007_001E"]
labels_l = [lbl for _, lbl in LANG_VARS]
vals_l   = [city[code] for code, _ in LANG_VARS]
pcts_l   = [v / total_lang * 100 for v in vals_l]

fig = go.Figure(go.Bar(
    x=labels_l, y=vals_l,
    marker_color=LANG_COLORS, marker_line_width=0,
    text=[f"{p:.1f}%" for p in pcts_l], textposition="outside",
    hovertemplate="<b>%{x}</b><br>%{y:,}  (%{text})<extra></extra>",
))
fig.update_layout(
    title="Language Spoken at Home · Lawrence, MA · ACS 2022",
    yaxis=dict(title="Population", gridcolor="#efefef"),
    plot_bgcolor="white", paper_bgcolor="white",
    height=400, width=680,
    margin=dict(l=60, r=40, t=70, b=80),
)
fig.show()

# English proficiency detail
print("English proficiency:")
limited_eng = city["B06007_004E"] + city["B06007_006E"] + city["B06007_008E"] + city["B06007_009E"]
print(f"  Limited English proficiency (est.): {limited_eng:,.0f}"
      f"  ({limited_eng/total_lang*100:.1f}% of residents 5+)")

English proficiency:
  Limited English proficiency (est.): 63,920  (78.7% of residents 5+)


## 7. Census-Tract Level Comparisons

All 18 Lawrence census tracts, comparing Hispanic share and median age.

In [8]:
def tract_label(name):
    if "Census Tract" in str(name):
        num = name.split(";")[0].replace("Census Tract", "").strip()
        return f"Tract {num}"
    return str(name)

tract = tract.copy()
tract["_label"]   = tract["NAME"].apply(tract_label)
tract["hisp_pct"] = tract["B03001_003E"] / tract["B03001_001E"] * 100
tract["total_pop"]= tract["B01001_001E"]
tract_sorted = tract.sort_values("hisp_pct", ascending=False)

fig = go.Figure()
fig.add_trace(go.Bar(
    name="Hispanic/Latino %",
    x=tract_sorted["_label"], y=tract_sorted["hisp_pct"],
    marker_color="#BE2D2C", marker_line_width=0,
    yaxis="y",
    hovertemplate="<b>%{x}</b><br>Hispanic/Latino: %{y:.1f}%<extra></extra>",
))
fig.add_trace(go.Scatter(
    name="Median Age",
    x=tract_sorted["_label"], y=tract_sorted["B01002_001E"],
    mode="lines+markers",
    marker=dict(color="#D4A724", size=8),
    line=dict(color="#D4A724", width=2),
    yaxis="y2",
    hovertemplate="<b>%{x}</b><br>Median Age: %{y:.1f}<extra></extra>",
))
fig.update_layout(
    title="Hispanic/Latino Share & Median Age by Tract · Lawrence, MA",
    xaxis=dict(title="", tickangle=-40),
    yaxis =dict(title="Hispanic/Latino (%)", gridcolor="#efefef", range=[0, 110]),
    yaxis2=dict(title="Median Age", overlaying="y", side="right", range=[15, 60]),
    legend=dict(orientation="h", y=1.06, x=0.5, xanchor="center"),
    plot_bgcolor="white", paper_bgcolor="white",
    height=500, width=920,
    margin=dict(l=60, r=60, t=80, b=130),
)
fig.show()

In [9]:
# Population by tract — sorted ascending for a horizontal bar
tract_pop = tract.sort_values("total_pop")

fig = go.Figure(go.Bar(
    y=tract_pop["_label"], x=tract_pop["total_pop"], orientation="h",
    marker_color="#1B3A6B", marker_line_width=0,
    text=tract_pop["total_pop"].apply(lambda v: f"{v:,.0f}"),
    textposition="outside",
    hovertemplate="<b>%{y}</b><br>Population: %{x:,}<extra></extra>",
))
fig.update_layout(
    title="Population by Census Tract · Lawrence, MA",
    xaxis=dict(title="Total Population", gridcolor="#efefef"),
    plot_bgcolor="white", paper_bgcolor="white",
    height=560, width=720,
    margin=dict(l=90, r=80, t=70, b=50),
)
fig.show()

## 8. Tract Summary Table

In [10]:
summary = pd.DataFrame({
    "Tract":           tract["_label"],
    "Population":      tract["total_pop"].astype(int),
    "Median Age":      tract["B01002_001E"].round(1),
    "Hispanic %":      (tract["B03001_003E"] / tract["B03001_001E"] * 100).round(1),
    "White %":         (tract["B02001_002E"] / tract["B02001_001E"] * 100).round(1),
    "Black %":         (tract["B02001_003E"] / tract["B02001_001E"] * 100).round(1),
    "English Only %":  (tract["B06007_002E"] / tract["B06007_001E"] * 100).round(1),
}).sort_values("Tract").reset_index(drop=True)

display(
    summary.style
    .background_gradient(subset=["Hispanic %"],      cmap="Reds",   vmin=0, vmax=100)
    .background_gradient(subset=["English Only %"],  cmap="Blues",  vmin=0, vmax=100)
    .background_gradient(subset=["Population"],      cmap="Greens")
    .format({"Population": "{:,}"})
)

,Tract,Population,Median Age,Hispanic %,White %,Black %,English Only %
0,Tract 2501,"3,809",38.300000,78.600000,28.500000,4.800000,25.100000
1,Tract 2502,"6,831",28.500000,84.300000,26.800000,2.900000,21.700000
2,Tract 2503,"2,504",35.500000,94.300000,15.900000,1.400000,10.000000
3,Tract 2504,"4,282",28.900000,94.300000,17.900000,4.200000,9.300000
4,Tract 2505,"4,392",25.900000,93.800000,22.200000,4.600000,10.800000
5,Tract 2506,"5,933",32.500000,84.400000,23.700000,7.400000,14.700000
6,Tract 2507,"5,484",31.100000,90.600000,19.400000,1.200000,9.200000
7,Tract 2508,"8,550",27.300000,78.700000,30.500000,5.500000,39.100000
8,Tract 2509,"2,425",32.800000,97.400000,19.800000,2.400000,4.500000
9,Tract 2510,"1,907",31.000000,93.100000,12.900000,4.000000,11.300000
